<a href="https://colab.research.google.com/github/johanhoffman/DD2365_FEniCSx/blob/port-stokes-amr/template-report-Stokes-AMR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **The Stokes equations - adaptive mesh refinement**
**Johan Hoffman**

# **Abstract**

This short report shows an example of how to use FEniCSx to solve the Stokes equations with goal-oriented adaptive mesh refinement (AMR) using the dual weighted residual (DWR) method, which is used in the course DD2365 Advanced Computation in Fluid Mechanics, at the KTH Royal Institute of Technology.

[DD2365 course website.](https://kth.instructure.com/courses/17071)

# **About the code**

In [ ]:
# This program is an example file for the course
# DD2365 Advanced Computation in Fluid Mechanics,
# KTH Royal Institute of Technology, Stockholm, Sweden.

# Copyright (C) 2020-2026 Johan Hoffman (jhoffman@kth.se)

# This file is part of the course DD2365 Advanced Computation in Fluid Mechanics
# KTH Royal Institute of Technology, Stockholm, Sweden
#
# This is free software: you can redistribute it and/or modify
# it under the terms of the GNU Lesser General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.

# This is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the
# GNU Lesser General Public License for more details.

# You should have received a copy of the GNU Lesser General Public License
# along with this. If not, see <http://www.gnu.org/licenses/>.

# **Set up environment**

In [ ]:
# --- canonical: bootstrap v1 ---
import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if not _on_colab:
    os.environ.setdefault("OMP_NUM_THREADS", "1")

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
import numpy as np
from mpi4py import MPI
import ufl
import basix.ufl as bufl
from dolfinx import fem
from dolfinx.fem import (functionspace, Function, assemble_scalar, assemble_vector,
                         form as fem_form)
from dolfinx.fem.petsc import NonlinearProblem

In [ ]:
# --- canonical: gmsh_rect_minus_circles v2 ---
import math
import gmsh
from mpi4py import MPI
from dolfinx.io import gmsh as gmshio


def gmsh_rect_minus_circles(L, H, circles, resolution):
    """Rectangle [0,L]×[0,H] minus circular holes, meshed with gmsh OCC.

    Parameters
    ----------
    L, H : float
        Rectangle dimensions.
    circles : list of (cx, cy, r)
        Circle centres and radii to subtract.
    resolution : int
        Mesh density parameter.  The global size bound is

            lc = 0.65 * sqrt(L² + H²) / resolution

        This matches the mshr/CGAL semantics used in the legacy FEniCS
        notebooks: mshr's ``resolution`` sets a CGAL size bound equal to
        the bounding-box diagonal divided by ``resolution``; gmsh realised
        edges are approximately 0.6× that bound.  The factor 0.65 was
        calibrated so that the standard test case (L=4, H=2, 3 circular
        holes, resolution=32) produces ≈2319 cells — matching the legacy
        mshr mesh (2319 cells, 1247 P1 dofs).

    Returns
    -------
    msh : dolfinx.mesh.Mesh
    cell_tags : dolfinx.mesh.MeshTags   (fluid domain tag = 10)

    Note
    ----
    Facet tags are not produced here.  Call ``tag_boundaries(msh, L, H)``
    on the final mesh (after any refinement) to obtain boundary MeshTags.
    """
    _ALPHA = 0.65
    lc = _ALPHA * math.sqrt(L**2 + H**2) / resolution

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)

    rect = gmsh.model.occ.addRectangle(0.0, 0.0, 0.0, L, H)
    disks = [(2, gmsh.model.occ.addDisk(cx, cy, 0.0, r, r)) for cx, cy, r in circles]
    if disks:
        gmsh.model.occ.cut([(2, rect)], disks)
    gmsh.model.occ.synchronize()

    gmsh.option.setNumber("Mesh.MeshSizeMax", lc)

    surfaces = gmsh.model.getEntities(2)
    gmsh.model.addPhysicalGroup(2, [s[1] for s in surfaces], tag=10, name="domain")

    gmsh.model.mesh.generate(2)

    mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, rank=0, gdim=2)
    gmsh.finalize()
    return mesh_data.mesh, mesh_data.cell_tags
# --- end canonical: gmsh_rect_minus_circles ---

In [ ]:
# --- canonical: refine_cells v2 ---
import numpy as np
import dolfinx.mesh
from dolfinx.mesh import RefinementOption


def refine_cells(msh, predicate_or_mask):
    """Refine selected cells using the Plaza algorithm.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    predicate_or_mask : callable or array-like of bool
        Either a callable ``f(midpoints) -> bool array`` where
        ``midpoints`` has shape ``(ncells, gdim)``, or a boolean array
        of length ``num_local_cells`` that directly marks which cells
        to refine (useful when marks come from an existing DG0 field).

    Returns
    -------
    refined_msh : dolfinx.mesh.Mesh
    parent_cells : np.ndarray[np.int32]  shape (num_refined_cells,)
    parent_facets : np.ndarray[np.int8]  shape (num_refined_cells,)
        Local parent-facet index per refined cell, or -1 for interior.
    """
    tdim = msh.topology.dim
    num_cells = msh.topology.index_map(tdim).size_local

    if callable(predicate_or_mask):
        msh.topology.create_entities(1)
        msh.topology.create_connectivity(tdim, 0)
        midpoints = dolfinx.mesh.compute_midpoints(
            msh, tdim, np.arange(num_cells, dtype=np.int32)
        )
        marked = np.where(predicate_or_mask(midpoints))[0].astype(np.int32)
    else:
        mask = np.asarray(predicate_or_mask, dtype=bool)
        marked = np.where(mask[:num_cells])[0].astype(np.int32)

    if len(marked) == 0:
        n = msh.topology.index_map(tdim).size_local
        return msh, np.arange(n, dtype=np.int32), np.full(n, -1, dtype=np.int8)

    msh.topology.create_entities(1)
    msh.topology.create_connectivity(tdim, 1)
    edges = dolfinx.mesh.compute_incident_entities(msh.topology, marked, tdim, 1)
    edges = np.unique(edges).astype(np.int32)
    return dolfinx.mesh.refine(
        msh, edges, option=RefinementOption.parent_cell_and_facet
    )
# --- end canonical: refine_cells ---

In [ ]:
# --- canonical: plot_helpers v2 ---
import numpy as np
import matplotlib.pyplot as plt
import basix.ufl as _bufl
from dolfinx.fem import functionspace, Function


def plot_mesh(msh, title="Mesh"):
    """Plot a 2-D triangular mesh using matplotlib triplot."""
    msh.topology.create_connectivity(msh.topology.dim, 0)
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.triplot(x[:, 0], x[:, 1], gdm, linewidth=0.3, color="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def _p1_scalar_space(msh):
    return functionspace(msh, ("Lagrange", 1))


def _scatter_p1_scalar(f1):
    """Scatter a P1 scalar Function values to geometry nodes.

    Returns (x, geometry_connectivity, node_values).
    """
    V = f1.function_space
    msh = V.mesh
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    ldm = V.dofmap.list
    vals = np.zeros(x.shape[0])
    vals[gdm.ravel()] = f1.x.array.real[ldm.ravel()]
    return x, gdm, vals


def plot_p1(u, title="Solution"):
    """Plot a P1 scalar Function using matplotlib tripcolor (Gouraud shading)."""
    plot_scalar(u, title=title)


def plot_scalar(f, title="Scalar field"):
    """Plot any scalar Lagrange Function via tripcolor (interpolates to P1 if needed)."""
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    if el.degree == 1 and el.reference_value_shape == ():
        f1 = f
    else:
        f1 = Function(_p1_scalar_space(msh))
        f1.interpolate(f)
    x, gdm, vals = _scatter_p1_scalar(f1)
    fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="gouraud")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def plot_vector(u, title="Vector field", quiver=True, quiver_stride=8):
    """Plot a 2-D vector Lagrange Function as colour map of |u| + optional quiver.

    Parameters
    ----------
    u : dolfinx.fem.Function   value shape (2,); any Lagrange degree
    quiver : bool              overlay subsampled arrows (default True)
    quiver_stride : int        take every N-th mesh node for arrows
    """
    msh = u.function_space.mesh
    gdim = msh.geometry.dim
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    npts = x.shape[0]

    # Interpolate to P1 vector so values align with geometry nodes
    V1v = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=(gdim,)))
    u1v = Function(V1v)
    u1v.interpolate(u)

    # Scatter: for a block-size-2 P1 space, dofmap.list gives block indices
    ldm = V1v.dofmap.list      # (ncells, 3)  — block indices
    arr = u1v.x.array.real     # length npts * gdim, interleaved per block
    ux = np.zeros(npts)
    uy = np.zeros(npts)
    ux[gdm.ravel()] = arr[ldm.ravel() * gdim + 0]
    uy[gdm.ravel()] = arr[ldm.ravel() * gdim + 1]
    mag = np.sqrt(ux**2 + uy**2)

    fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, mag, shading="gouraud", cmap="viridis")
    plt.colorbar(tc, ax=ax, label="|u|")
    if quiver:
        idx = np.arange(0, npts, quiver_stride)
        ax.quiver(x[idx, 0], x[idx, 1], ux[idx], uy[idx],
                  color="white", alpha=0.6, width=0.002, scale_units="xy", scale=2.0)
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
# --- end canonical: plot_helpers ---

In [ ]:
# --- canonical: export_xdmf v2 ---
import sys
import pathlib
import basix.ufl as _bufl
from mpi4py import MPI
from dolfinx.io import XDMFFile
from dolfinx.fem import functionspace, Function


def _to_p1(f):
    """Interpolate f to P1 (scalar or vector) if needed.

    XDMF only reliably stores P1 Lagrange data; higher-degree functions (e.g.
    P2 velocity from Taylor-Hood) must be projected to P1 before export so that
    ParaView can read the file without geometry/value mismatches.
    """
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    vshape = el.reference_value_shape  # () for scalar, (2,) for vector
    if el.degree == 1:
        return f
    if vshape == ():
        V1 = functionspace(msh, ("Lagrange", 1))
    else:
        V1 = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=vshape))
    f1 = Function(V1, name=f.name)
    f1.interpolate(f)
    return f1


def export_xdmf(path, funcs, download=False):
    """Export a list of Functions to XDMF for ParaView.

    Higher-degree functions (P2, mixed sub-functions, etc.) are automatically
    interpolated to P1 before writing — XDMF stores values at mesh nodes and
    ParaView cannot reliably read nodal data for higher-order Lagrange elements.

    Parameters
    ----------
    path : str or Path
        Output .xdmf file path (an .h5 sidecar is written alongside it).
    funcs : list of dolfinx.fem.Function
        Functions to export (must share the same mesh).
    download : bool, optional
        If True *and* running on Colab, tar the .xdmf/.h5 pair and trigger
        a browser download.  Default False.
    """
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    funcs_p1 = [_to_p1(f) for f in funcs]
    with XDMFFile(MPI.COMM_WORLD, str(path), "w") as xdmf:
        if funcs_p1:
            xdmf.write_mesh(funcs_p1[0].function_space.mesh)
        for f in funcs_p1:
            xdmf.write_function(f)
    if download and "google.colab" in sys.modules:
        import tarfile
        from google.colab import files as colab_files
        tar_path = str(path.with_suffix(".tar.gz"))
        with tarfile.open(tar_path, "w:gz") as tar:
            tar.add(str(path), arcname=path.name)
            h5 = path.with_suffix(".h5")
            if h5.exists():
                tar.add(str(h5), arcname=h5.name)
        colab_files.download(tar_path)
# --- end canonical: export_xdmf ---

In [ ]:
# --- canonical: tag_boundaries v1 ---
import numpy as np
from dolfinx.mesh import locate_entities_boundary, meshtags


def tag_boundaries(msh, L, H, eps=None):
    """Tag exterior boundary facets of a [0,L]×[0,H] rectangle with holes.

    Assigns integer tags to all exterior boundary facets:

        left=1  (x ≈ 0),   right=2 (x ≈ L),
        lower=3 (y ≈ 0),   upper=4 (y ≈ H),
        objects=5 (remaining exterior facets — circle boundaries).

    Corners are unambiguous: boundary facets are edges, not vertices, so a
    corner vertex is shared by one vertical and one horizontal edge — each
    edge belongs to exactly one group and no facet is tagged twice.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    L, H : float
        Rectangle dimensions.
    eps : float, optional
        Coordinate tolerance.  Default ``1e-6 * max(L, H)``.

    Returns
    -------
    dolfinx.mesh.MeshTags
        Must be recomputed whenever the mesh changes (e.g. after refinement).
    """
    if eps is None:
        eps = 1e-6 * max(L, H)

    fdim = msh.topology.dim - 1
    msh.topology.create_entities(fdim)
    msh.topology.create_connectivity(fdim, msh.topology.dim)

    left  = locate_entities_boundary(msh, fdim, lambda x: x[0] <= eps)
    right = locate_entities_boundary(msh, fdim, lambda x: x[0] >= L - eps)
    lower = locate_entities_boundary(msh, fdim, lambda x: x[1] <= eps)
    upper = locate_entities_boundary(msh, fdim, lambda x: x[1] >= H - eps)

    known = np.unique(np.concatenate([left, right, lower, upper]))
    all_bdry = locate_entities_boundary(
        msh, fdim, lambda x: np.ones(x.shape[1], dtype=bool)
    )
    objects = np.setdiff1d(all_bdry, known)

    indices = np.concatenate([left, right, lower, upper, objects]).astype(np.int32)
    values  = np.concatenate([
        np.full(len(left),    1, dtype=np.int32),
        np.full(len(right),   2, dtype=np.int32),
        np.full(len(lower),   3, dtype=np.int32),
        np.full(len(upper),   4, dtype=np.int32),
        np.full(len(objects), 5, dtype=np.int32),
    ])
    order = np.argsort(indices)
    return meshtags(msh, fdim, indices[order], values[order])
# --- end canonical: tag_boundaries ---

# **Introduction**

\
The Stokes equations take the form

$\nabla p - \Delta u = f, \quad \nabla \cdot u = 0$

together with suitable boundary conditions.

Goal-oriented adaptive mesh refinement is based on the dual weighted residual (DWR) method. Given a quantity of interest (QoI) $\psi(u)$, one solves an adjoint (dual) problem to obtain an adjoint solution $(\phi, \theta)$. The adjoint solution acts as a weight in the error representation:

$\psi(u) - \psi(u_h) \approx \sum_K E_K$

where the cell error indicators are

$E_K = \int_K \bigl( f \cdot \phi + p_h \nabla \cdot \phi - \nabla u_h : \nabla \phi - \nabla \cdot u_h \, \theta \bigr) \, dx$

Cells with $|E_K| > \overline{|E|}$ (mean) are marked for refinement. After one AMR step, the refined mesh concentrates resolution where the error contribution is largest.

Here we use drag force on the circular obstacle as the QoI, encoded by the adjoint boundary condition $\phi = \phi_3 = (1, 0)$ on the circle boundary (tag 5).\


# **Method**

**Define domain and mesh**

In [ ]:
# Domain parameters
L = 4
H = 4
resolution = 32

circles = [(0.5, 0.5 * H, 0.2)]

# Generate mesh: rectangle minus circular hole
msh, cell_tags = gmsh_rect_minus_circles(L, H, circles, resolution)

# Optional initial uniform refinement (default: none)
init_no_levels = 0
for _ in range(init_no_levels):
    msh, _, _ = refine_cells(
        msh,
        lambda pts: np.sqrt((pts[:, 0] - 0.5)**2 + (pts[:, 1] - 2.0)**2) < 1.0,
    )

facet_tags = tag_boundaries(msh, L, H)
n_cells_init = msh.topology.index_map(msh.topology.dim).size_global
plot_mesh(msh, title=f"Initial mesh: {n_cells_init} cells")

**Define finite element approximation spaces**

In [ ]:
gdim = msh.geometry.dim

# Primal: Taylor-Hood P2/P1
P2v = bufl.element("Lagrange", msh.basix_cell(), 2, shape=(gdim,))
P1s = bufl.element("Lagrange", msh.basix_cell(), 1)
W   = functionspace(msh, bufl.mixed_element([P2v, P1s]))

w       = Function(W)
u_s, p_s = ufl.split(w)
v, q    = ufl.TestFunctions(W)

# Adjoint: P3/P2 (enriched for superconvergence of error representation)
P3v = bufl.element("Lagrange", msh.basix_cell(), 3, shape=(gdim,))
P2s = bufl.element("Lagrange", msh.basix_cell(), 2)
Wa  = functionspace(msh, bufl.mixed_element([P3v, P2s]))

wa         = Function(Wa)
phi_s, theta_s = ufl.split(wa)
va, qa     = ufl.TestFunctions(Wa)

Vc, _ = W.sub(0).collapse()
Qc, _ = W.sub(1).collapse()
Va, _ = Wa.sub(0).collapse()
Qa, _ = Wa.sub(1).collapse()
print(f"Primal dofs (V+Q) = {Vc.dofmap.index_map.size_global * Vc.dofmap.index_map_bs + Qc.dofmap.index_map.size_global}")
print(f"Adjoint dofs (V+Q) = {Va.dofmap.index_map.size_global * Va.dofmap.index_map_bs + Qa.dofmap.index_map.size_global}")

**Define boundary conditions**

Facet tags from `tag_boundaries(msh, L, H)`: left (inflow)=1, right (outflow)=2,
lower wall=3, upper wall=4, circle (body)=5.  Boundary conditions are imposed
weakly with $\gamma = C/h$, $C = 10^3$:

- Inflow (tag 1): $u = u_{\mathrm{in}}$, parabolic profile $u_{\mathrm{in}} = (4y(H-y)/H^2, 0)$
- Walls (tags 3, 4): $u = 0$
- Body (tag 5): $u = 0$; adjoint $\phi = \phi_3 = (1, 0)$ (drag QoI)
- Outflow (tag 2): do-nothing (no boundary term)

# **Results**

**Define and solve primal and adjoint problems**

In [ ]:
dx = ufl.Measure("dx", domain=msh)
ds = ufl.Measure("ds", domain=msh, subdomain_data=facet_tags)
h  = ufl.CellDiameter(msh)
x  = ufl.SpatialCoordinate(msh)

C     = 1.0e3
gamma = C / h

uin_v = ufl.as_vector([4.0 * x[1] * (H - x[1]) / (H * H), 0.0])
f_ufl = ufl.as_vector([0.0, 0.0])

# Primal residual form r((u,p); (v,q)) = 0
res = (
    - p_s * ufl.div(v) * dx
    + ufl.inner(ufl.grad(u_s), ufl.grad(v)) * dx
    + ufl.div(u_s) * q * dx
    - ufl.inner(f_ufl, v) * dx
    + gamma * ufl.inner(u_s - uin_v, v) * ds(1)  # ib: u = uin
    + gamma * ufl.inner(u_s, v) * ds(3)           # wb bottom: u = 0
    + gamma * ufl.inner(u_s, v) * ds(4)           # wb top: u = 0
    + gamma * ufl.inner(u_s, v) * ds(5)           # bb circle: u = 0
)
problem = NonlinearProblem(
    res, w,
    petsc_options_prefix="primal_",
    petsc_options={
        "snes_type": "newtonls", "snes_rtol": 1e-8, "snes_atol": 1e-10,
        "ksp_type": "preonly", "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
    },
)
w = problem.solve()
assert problem.solver.getConvergedReason() > 0, "Primal Newton did not converge"
u_s, p_s = ufl.split(w)
norm_u = assemble_scalar(fem_form(ufl.inner(u_s, u_s) * dx))**0.5
print(f"Primal converged.  ||u||_L2 = {norm_u:.6f}")

In [ ]:
# Adjoint residual form (QoI: drag on circle => phi=phi3=(1,0) on tag 5)
phi3 = ufl.as_vector([1.0, 0.0])

res_a = (
    - qa * ufl.div(phi_s) * dx
    + ufl.inner(ufl.grad(va), ufl.grad(phi_s)) * dx
    + ufl.div(va) * theta_s * dx
    + gamma * ufl.inner(phi_s, va) * ds(1)              # ib: phi = 0
    + gamma * ufl.inner(phi_s, va) * ds(3)              # wb: phi = 0
    + gamma * ufl.inner(phi_s, va) * ds(4)              # wb: phi = 0
    + gamma * ufl.inner(phi_s - phi3, va) * ds(5)       # bb: phi = phi3
)
problem_a = NonlinearProblem(
    res_a, wa,
    petsc_options_prefix="adjoint_",
    petsc_options={
        "snes_type": "newtonls", "snes_rtol": 1e-8, "snes_atol": 1e-10,
        "ksp_type": "preonly", "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
    },
)
wa = problem_a.solve()
assert problem_a.solver.getConvergedReason() > 0, "Adjoint Newton did not converge"
phi_s, theta_s = ufl.split(wa)
norm_phi = assemble_scalar(fem_form(ufl.inner(phi_s, phi_s) * dx))**0.5
print(f"Adjoint converged. ||phi||_L2 = {norm_phi:.6f}")

**Compute error indicators and refine mesh**

In [ ]:
# DG0 test function gives one indicator per cell
WDG = functionspace(msh, ("DG", 0))
elm = ufl.TestFunction(WDG)

local_error_ufl = (
    elm * ufl.inner(f_ufl, phi_s) * dx
    + elm * p_s * ufl.div(phi_s) * dx
    - elm * ufl.inner(ufl.grad(u_s), ufl.grad(phi_s)) * dx
    - elm * ufl.div(u_s) * theta_s * dx
)
err_b  = assemble_vector(fem_form(local_error_ufl))
E_raw  = err_b.array
dm     = WDG.dofmap.list   # (ncells, 1) — do not assume identity
E_cells = E_raw[dm.ravel()]
E_abs  = np.abs(E_cells)
E_mean = E_abs.mean()
n_marked = int((E_abs > E_mean).sum())
print(f"Error indicators: mean |E_K| = {E_mean:.4e},  marked = {n_marked}/{n_cells_init}")

# Total error estimate (includes penalty boundary terms)
tot_err_form = fem_form(
    ufl.inner(f_ufl, phi_s) * dx
    + p_s * ufl.div(phi_s) * dx
    - ufl.inner(ufl.grad(u_s), ufl.grad(phi_s)) * dx
    - ufl.div(u_s) * theta_s * dx
    - gamma * ufl.inner(u_s - uin_v, phi_s) * ds(1)
    - gamma * ufl.inner(u_s, phi_s) * ds(3)
    - gamma * ufl.inner(u_s, phi_s) * ds(4)
    - gamma * ufl.inner(u_s, phi_s) * ds(5)
)
tot_err = assemble_scalar(tot_err_form)
print(f"tot_err = {tot_err:.6e}")

# AMR: refine cells where |E_K| > mean
marked_mask = E_abs > E_mean
msh_ref, _, _ = refine_cells(msh, marked_mask)
facet_tags_ref = tag_boundaries(msh_ref, L, H)
n_cells_ref = msh_ref.topology.index_map(msh_ref.topology.dim).size_global
print(f"Refined mesh: {n_cells_ref} cells  (was {n_cells_init})")
plot_mesh(msh_ref, title=f"Refined mesh: {n_cells_ref} cells")

**Visualize solution and export files**

In [ ]:
# Extract collapsed Functions for plotting and export
u_h     = w.sub(0).collapse();   u_h.name = "u"
p_h     = w.sub(1).collapse();   p_h.name = "p"
phi_h   = wa.sub(0).collapse();  phi_h.name = "phi"
theta_h = wa.sub(1).collapse();  theta_h.name = "theta"

plot_vector(u_h,     title="Velocity field |u|")
plot_scalar(p_h,     title="Pressure field p")
plot_vector(phi_h,   title="Adjoint velocity |phi|")
plot_scalar(theta_h, title="Adjoint pressure theta")

export_xdmf("results-Stokes-AMR/stokes_amr.xdmf", [u_h, p_h, phi_h, theta_h])
# On Colab, pass download=True to trigger a browser download:
# export_xdmf("results-Stokes-AMR/stokes_amr.xdmf", [u_h, p_h, phi_h, theta_h], download=True)

In [ ]:
# Quantities of interest
u_s2, p_s2       = ufl.split(w)
phi_s2, theta_s2 = ufl.split(wa)

norm_u     = assemble_scalar(fem_form(ufl.inner(u_s2, u_s2) * dx))**0.5
norm_p     = assemble_scalar(fem_form(p_s2**2 * dx))**0.5
norm_phi   = assemble_scalar(fem_form(ufl.inner(phi_s2, phi_s2) * dx))**0.5
norm_theta = assemble_scalar(fem_form(theta_s2**2 * dx))**0.5

Vc_p, _ = W.sub(0).collapse()
Qc_p, _ = W.sub(1).collapse()
Vc_a, _ = Wa.sub(0).collapse()
Qc_a, _ = Wa.sub(1).collapse()
V_primal = Vc_p.dofmap.index_map.size_global * Vc_p.dofmap.index_map_bs
Q_primal = Qc_p.dofmap.index_map.size_global
V_adj    = Vc_a.dofmap.index_map.size_global * Vc_a.dofmap.index_map_bs
Q_adj    = Qc_a.dofmap.index_map.size_global

print(f"||u||_L2              = {norm_u:.6f}")
print(f"||p||_L2              = {norm_p:.6f}")
print(f"||phi||_L2            = {norm_phi:.6f}")
print(f"||theta||_L2          = {norm_theta:.6f}")
print(f"tot_err               = {tot_err:.6e}")
print(f"mean |E_K|            = {E_mean:.6e}")
print(f"marked cells          = {n_marked}")
print(f"cells (initial)       = {n_cells_init}")
print(f"cells (refined)       = {n_cells_ref}")
print(f"primal dofs (V+Q)     = {V_primal + Q_primal}  (V={V_primal}, Q={Q_primal})")
print(f"adjoint dofs (V+Q)    = {V_adj + Q_adj}  (V={V_adj}, Q={Q_adj})")

# **Discussion**

An adaptive finite element method was implemented in FEniCSx to solve the Stokes equations in 2D. An adjoint problem was defined and solved to obtain cell-wise error indicators via the dual weighted residual method. Cells where the local error indicator exceeded the mean were marked for refinement. One AMR step was performed, producing a refined mesh that concentrates resolution near the circular obstacle where the drag-force QoI is most sensitive to the local flow.